# Anomaly Detection with One-Class SVMs

**Course:** Machine Learning and Analytics (SUTD)  
**Duration:** ~40 minutes  
**Type:** Interactive Teaching Notebook

---

## What You Will Learn

1. Why standard classification struggles with highly imbalanced data (the *accuracy paradox*)
2. The intuition and theory behind **One-Class SVMs** for anomaly detection
3. How to train a One-Class SVM using only "normal" data
4. How to evaluate anomaly detectors with appropriate metrics
5. The effect of hyperparameters (`nu`, `gamma`) on the decision boundary

---

### Key Question

> *If we only know what **normal** data looks like, how do we detect outliers?*

In many real-world problems (fraud detection, network intrusion, manufacturing defects), anomalies are **rare** and **diverse**. Collecting labelled anomaly examples is expensive or impossible. One-Class SVMs offer an elegant solution: learn a **tight boundary** around the normal data, and flag everything outside as anomalous.

---

## 1. Setup & Packages

We begin by importing the libraries we will use throughout this notebook. Make sure you have `scikit-learn`, `matplotlib`, `seaborn`, `numpy`, and `pandas` installed.

In [ ]:
# Standard libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs, make_moons
from sklearn import metrics

# Add parent directory so we can import our project utilities
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.data import generate_fraud_dataset, prepare_data_splits
from utils.data import generate_toy_2d, generate_moons_with_anomalies

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports successful!')

---

## 2. Understanding Our Data

We will work with a **synthetic fraud dataset** that mimics credit card transactions. The dataset has 10 features (transaction amount, time of day, frequency, distance from home, etc.) and a binary label: `0 = normal`, `1 = fraud`.

The fraud rate is intentionally low (~5%), reflecting real-world class imbalance.

In [ ]:
# Generate the synthetic fraud dataset
X_df, y = generate_fraud_dataset(n_samples=5000, anomaly_ratio=0.05, random_state=42)

print(f'Dataset shape: {X_df.shape}')
print(f'Features: {list(X_df.columns)}')
print(f'\nTarget distribution:')
print(y.value_counts())
print(f'\nFraud ratio: {y.mean():.2%}')

In [ ]:
# Quick look at the data
X_df.head(10)

In [ ]:
# Summary statistics
X_df.describe().round(2)

### Exercise 1

**Task:** Plot histograms for three features (`amount`, `distance_from_home`, `merchant_risk_score`), comparing the distributions of **normal** vs **fraud** transactions. Use side-by-side or overlaid histograms with different colors.

*Hint:* Separate the data by label, then use `plt.hist()` or `sns.histplot()` with the `hue` parameter.

In [ ]:
# ============================================================
# EXERCISE 1: Compare feature distributions (normal vs fraud)
# TODO: Create a figure with 3 subplots (1 row, 3 columns)
# TODO: For each feature in ['amount', 'distance_from_home', 'merchant_risk_score']:
#       - Plot overlaid histograms for normal (y==0) and fraud (y==1)
#       - Use alpha=0.5 for transparency, different colors, and a legend
# ============================================================

features_to_plot = ['amount', 'distance_from_home', 'merchant_risk_score']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, features_to_plot):
    # TODO: Plot histogram for normal transactions (y == 0)
    # ax.hist(______, bins=40, alpha=0.5, label='Normal', color='steelblue')
    
    # TODO: Plot histogram for fraud transactions (y == 1)
    # ax.hist(______, bins=40, alpha=0.5, label='Fraud', color='crimson')
    
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# PCA visualization: project all features down to 2D
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X_df)

pca_viz = PCA(n_components=2)
X_pca = pca_viz.fit_transform(X_scaled_viz)

plt.figure(figsize=(10, 7))
normal_mask = y == 0
plt.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1],
            c='steelblue', alpha=0.3, s=15, label='Normal')
plt.scatter(X_pca[~normal_mask, 0], X_pca[~normal_mask, 1],
            c='crimson', alpha=0.8, s=40, marker='x', label='Fraud')
plt.xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA Projection of Fraud Dataset')
plt.legend()
plt.show()

---

## 3. The Accuracy Paradox

Before jumping to One-Class SVMs, let us see what happens when we naively apply a standard classifier to highly imbalanced data.

**The accuracy paradox:** A model that predicts *everything as normal* achieves ~95% accuracy when only 5% of data is anomalous. This sounds great but is completely useless -- it catches zero fraud!

Let us demonstrate this with a simple `LogisticRegression`.

In [ ]:
# Standard train/test split (using BOTH classes in training)
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_df, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler_lr = StandardScaler()
X_train_lr_s = scaler_lr.fit_transform(X_train_lr)
X_test_lr_s = scaler_lr.transform(X_test_lr)

# Train a simple logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_lr_s, y_train_lr)

y_pred_lr = lr_model.predict(X_test_lr_s)
print('Logistic Regression (standard training):')
print(f'  Accuracy:  {metrics.accuracy_score(y_test_lr, y_pred_lr):.4f}')
print(f'  Precision: {metrics.precision_score(y_test_lr, y_pred_lr):.4f}')
print(f'  Recall:    {metrics.recall_score(y_test_lr, y_pred_lr):.4f}')
print(f'  F1:        {metrics.f1_score(y_test_lr, y_pred_lr):.4f}')

### Exercise 2

**Task:** Compute and print the accuracy, precision, recall, and F1-score for a "dummy" baseline that predicts **all samples as normal** (i.e., `y_pred = 0` for every test sample). Then answer the question below.

**Question:** *Why is accuracy misleading here? Which metric drops to zero for the dummy baseline, and why does that matter?*

In [ ]:
# ============================================================
# EXERCISE 2: The accuracy paradox
# TODO: Create a dummy prediction array of all zeros (same length as y_test_lr)
# TODO: Compute accuracy, precision, recall, and F1 for this dummy baseline
# ============================================================

# TODO: Create dummy predictions (predict everything as normal = 0)
# y_pred_dummy = ____

# TODO: Compute and print the four metrics
# accuracy_dummy  = metrics.accuracy_score(____)
# precision_dummy = metrics.precision_score(____)
# recall_dummy    = metrics.recall_score(____)
# f1_dummy        = metrics.f1_score(____)

# print(f'Dummy Baseline (predict all normal):')
# print(f'  Accuracy:  {accuracy_dummy:.4f}')
# print(f'  Precision: {precision_dummy:.4f}')
# print(f'  Recall:    {recall_dummy:.4f}')
# print(f'  F1:        {f1_dummy:.4f}')

# ANSWER: Why is accuracy misleading here?
# (Write your answer as a comment)
# ____

---

## 4. One-Class SVM Theory

### The Core Idea

A **One-Class SVM** learns a decision boundary that encloses the *normal* data, without ever seeing labelled anomalies. It answers the question:

> *Given only examples of "normal," can we learn what normal looks like and flag anything that deviates?*

### How It Works

1. **Map data to a high-dimensional feature space** using a kernel (typically RBF).
2. **Find the smallest hypersphere** (or maximum-margin hyperplane from the origin) that encloses most of the training data.
3. Points **inside** the boundary are classified as normal (+1); points **outside** are anomalies (-1).

### Mathematical Formulation (High-Level)

The One-Class SVM (Scholkopf et al., 2001) solves:

$$\min_{\mathbf{w}, \rho, \xi} \frac{1}{2} \|\mathbf{w}\|^2 + \frac{1}{\nu n} \sum_{i=1}^{n} \xi_i - \rho$$

subject to:

$$\mathbf{w} \cdot \Phi(\mathbf{x}_i) \geq \rho - \xi_i, \quad \xi_i \geq 0$$

where:
- $\Phi(\cdot)$ is the kernel-induced feature map
- $\rho$ is the offset of the hyperplane from the origin
- $\xi_i$ are slack variables allowing some points to fall outside the boundary
- $\nu \in (0, 1]$ is a key hyperparameter (see below)

### Key Hyperparameters

| Parameter | Role | Typical Values |
|-----------|------|----------------|
| `kernel`  | Defines the feature space mapping. RBF is the most common choice. | `'rbf'` |
| `nu`      | Upper bound on the fraction of training errors (outliers) and lower bound on the fraction of support vectors. Think of it as "how much of the training data am I willing to let fall outside the boundary." | 0.01 -- 0.5 |
| `gamma`   | Controls the RBF kernel width. Large gamma = tight, local boundary; small gamma = smooth, global boundary. | `'scale'`, 0.01 -- 10 |

### Visual Intuition

Imagine your normal data as a cloud of points. The One-Class SVM draws a **flexible boundary** (thanks to the RBF kernel) that hugs the cloud tightly. Any new point that falls outside this boundary is flagged as an anomaly.

- **Small `nu`** = very tight boundary (few outliers allowed) = sensitive to anomalies
- **Large `nu`** = looser boundary = more tolerant
- **Large `gamma`** = boundary follows every bump in the data (risk of overfitting)
- **Small `gamma`** = smooth, globular boundary (risk of underfitting)

---

## 5. Training a One-Class SVM

### Important: Train on Normal Data Only!

Unlike a standard classifier, the One-Class SVM is trained using **only** normal samples. This is its key strength -- we do not need labelled anomalies for training.

Our `prepare_data_splits` utility:
1. Separates normal and anomaly samples
2. Creates a training set from normal data only
3. Creates a test set with both normal and anomaly samples
4. Standardizes the features

In [ ]:
# Prepare data: training set = normal only, test set = both
X_train, X_test, y_test, scaler = prepare_data_splits(X_df, y, test_size=0.3, random_state=42)

print(f'Training set: {X_train.shape[0]} samples (all normal)')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'  - Normal in test: {(y_test == 0).sum()}')
print(f'  - Fraud in test:  {(y_test == 1).sum()}')

### Exercise 3

**Task:** Complete the code below to create, train, and use a One-Class SVM. Fill in each `TODO`.

In [ ]:
# ============================================================
# EXERCISE 3: Train a One-Class SVM
# ============================================================

from sklearn.svm import OneClassSVM

# TODO: Create a OneClassSVM with kernel='rbf', nu=0.1, gamma=0.1
model = ____

# TODO: Fit the model on X_train (normal data only)
____

# TODO: Predict on X_test (-1 = anomaly, 1 = normal)
y_pred = ____

print(f'Predictions: {np.unique(y_pred, return_counts=True)}')
print(f'Number of detected anomalies: {(y_pred == -1).sum()}')
print(f'Number of detected normals:   {(y_pred == 1).sum()}')

In [ ]:
# --- SOLUTION (try the exercise above first!) ---
#
# model = OneClassSVM(kernel='rbf', nu=0.1, gamma=0.1)
# model.fit(X_train)
# y_pred = model.predict(X_test)
#
# print(f'Predictions: {np.unique(y_pred, return_counts=True)}')
# print(f'Number of detected anomalies: {(y_pred == -1).sum()}')
# print(f'Number of detected normals:   {(y_pred == 1).sum()}')

---

## 6. Evaluation

### Choosing the Right Metrics

For anomaly detection, we care about:

| Metric | What it measures |
|--------|------------------|
| **Precision** | Of all samples flagged as anomalies, how many are truly anomalous? |
| **Recall (Sensitivity)** | Of all true anomalies, how many did we catch? |
| **F1-Score** | Harmonic mean of precision and recall |
| **ROC-AUC** | Area under the ROC curve (uses continuous decision scores) |
| **PR-AUC** | Area under the Precision-Recall curve (better for imbalanced data) |
| **Confusion Matrix** | Full breakdown of TP, FP, TN, FN |

**Note:** The One-Class SVM outputs `-1` for anomaly and `1` for normal. We need to convert this to match our labels (`1` = fraud, `0` = normal) before computing metrics.

In [ ]:
# If you completed Exercise 3, use your model. Otherwise, train one here:
model = OneClassSVM(kernel='rbf', nu=0.1, gamma=0.1)
model.fit(X_train)
y_pred_raw = model.predict(X_test)

# Convert One-Class SVM output to our label convention:
# SVM: -1 = anomaly, +1 = normal  -->  Our labels: 1 = fraud, 0 = normal
y_pred_binary = (y_pred_raw == -1).astype(int)

# Decision function scores (more negative = more anomalous)
decision_scores = model.decision_function(X_test)
# Negate so that higher = more anomalous (for ROC/PR curves)
anomaly_scores = -decision_scores

# Print classification report
print('Classification Report:')
print(metrics.classification_report(y_test, y_pred_binary,
                                     target_names=['Normal', 'Fraud']))

# Key metrics
roc_auc = metrics.roc_auc_score(y_test, anomaly_scores)
pr_auc = metrics.average_precision_score(y_test, anomaly_scores)
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR-AUC:  {pr_auc:.4f}')

In [ ]:
# Plot ROC curve, Precision-Recall curve, and Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ROC Curve
fpr, tpr, _ = metrics.roc_curve(y_test, anomaly_scores)
axes[0].plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC-AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')

# Precision-Recall Curve
precision_curve, recall_curve, _ = metrics.precision_recall_curve(y_test, anomaly_scores)
axes[1].plot(recall_curve, precision_curve, color='green', lw=2,
             label=f'PR-AUC = {pr_auc:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')

# Confusion Matrix
cm = metrics.confusion_matrix(y_test, y_pred_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

### Exercise 4

**Task:** Re-train the One-Class SVM with `nu = 0.05` and `nu = 0.2` (keeping `gamma = 0.1`). For each, compute the precision, recall, F1, and ROC-AUC. Fill in the comparison table below.

| `nu` | Precision | Recall | F1 | ROC-AUC |
|------|-----------|--------|----|---------|
| 0.05 | ? | ? | ? | ? |
| 0.10 | (from above) | (from above) | (from above) | (from above) |
| 0.20 | ? | ? | ? | ? |

**Question:** As `nu` increases, what happens to precision and recall? Why?

In [ ]:
# ============================================================
# EXERCISE 4: Compare different nu values
# TODO: For each nu in [0.05, 0.1, 0.2]:
#       1. Create and fit a OneClassSVM (kernel='rbf', gamma=0.1)
#       2. Predict on X_test and convert to binary labels
#       3. Compute precision, recall, F1, and ROC-AUC
#       4. Print or store the results
# ============================================================

nu_values = [0.05, 0.1, 0.2]
results = []

for nu in nu_values:
    # TODO: Create and fit the model
    # model_nu = OneClassSVM(kernel='rbf', nu=____, gamma=0.1)
    # model_nu.fit(____)
    
    # TODO: Predict and convert labels
    # y_pred_nu = ____
    # y_pred_nu_binary = (y_pred_nu == -1).astype(int)
    
    # TODO: Compute anomaly scores and metrics
    # anomaly_scores_nu = -model_nu.decision_function(X_test)
    # prec = metrics.precision_score(y_test, y_pred_nu_binary)
    # rec  = metrics.recall_score(y_test, y_pred_nu_binary)
    # f1   = metrics.f1_score(y_test, y_pred_nu_binary)
    # auc  = metrics.roc_auc_score(y_test, anomaly_scores_nu)
    
    # results.append({'nu': nu, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': auc})
    pass

# TODO: Display results as a DataFrame
# pd.DataFrame(results).round(4)

---

## 7. Visualizing the Decision Boundary in 2D

To build intuition, let us work with 2D data where we can directly visualize the decision boundary. We will use a **two-moons** shaped dataset with scattered anomalies.

This shows that the One-Class SVM (with an RBF kernel) can learn **non-linear** boundaries.

In [ ]:
# Generate 2D moon-shaped data with anomalies
X_2d, y_2d = generate_moons_with_anomalies(
    n_normal=400, n_anomaly=40, noise=0.1, random_state=42
)

# Separate normal and anomaly for visualization
X_normal_2d = X_2d[y_2d == 1]
X_anomaly_2d = X_2d[y_2d == -1]

print(f'Normal points: {len(X_normal_2d)}')
print(f'Anomaly points: {len(X_anomaly_2d)}')

In [ ]:
# Train One-Class SVM on normal data only (2D case)
oc_svm_2d = OneClassSVM(kernel='rbf', nu=0.1, gamma=2.0)
oc_svm_2d.fit(X_normal_2d)

# Create a mesh grid for plotting the decision boundary
xx, yy = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 300),
    np.linspace(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 300)
)
Z = oc_svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
plt.contourf(xx, yy, Z, levels=[0, Z.max()], colors='palegreen', alpha=0.3)
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkgreen')

plt.scatter(X_normal_2d[:, 0], X_normal_2d[:, 1],
            c='steelblue', s=20, alpha=0.5, label='Normal')
plt.scatter(X_anomaly_2d[:, 0], X_anomaly_2d[:, 1],
            c='crimson', s=60, marker='x', linewidths=2, label='Anomaly')

plt.title('One-Class SVM Decision Boundary (2D Moons Data)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.show()

### Exercise 5

**Task:** Write a reusable function `plot_decision_boundary` that takes a fitted One-Class SVM model, 2D data (X, y), and plots the decision boundary with data points. Complete the TODOs in the template below.

In [ ]:
# ============================================================
# EXERCISE 5: Write a reusable decision boundary plotting function
# ============================================================

def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    """
    Plot the decision boundary of a fitted One-Class SVM on 2D data.
    
    Parameters
    ----------
    model : fitted OneClassSVM
    X : array of shape (n_samples, 2)
    y : array of labels (1 = normal, -1 = anomaly)
    title : str
    ax : matplotlib Axes (optional)
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    # TODO: Create a mesh grid covering the data range (with some margin)
    # margin = 1.0
    # xx, yy = np.meshgrid(
    #     np.linspace(X[:, 0].min() - margin, X[:, 0].max() + margin, 200),
    #     np.linspace(____)
    # )
    
    # TODO: Compute decision function values on the grid
    # Z = model.decision_function(____)
    # Z = Z.reshape(xx.shape)
    
    # TODO: Plot filled contours for the anomaly region (Z < 0)
    # ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
    
    # TODO: Plot the normal region and the decision boundary contour line
    # ax.contourf(____)
    # ax.contour(____)
    
    # TODO: Scatter plot the normal and anomaly points with different markers/colors
    # normal_mask = y == 1
    # ax.scatter(____)
    # ax.scatter(____)
    
    ax.set_title(title)
    ax.legend()
    
    return ax


# Test your function (uncomment after completing):
# plot_decision_boundary(oc_svm_2d, X_2d, y_2d, title='My Decision Boundary Plot')
# plt.show()

---

## 8. Hyperparameter Exploration

The two most important hyperparameters for One-Class SVM with an RBF kernel are:

- **`gamma`** -- controls the RBF kernel width. It determines how "local" the boundary is.
  - Small gamma: smooth, wide boundary (may underfit)
  - Large gamma: tight, complex boundary (may overfit)

- **`nu`** -- upper bound on the fraction of outliers in the training data.
  - Small nu: very few training points allowed outside the boundary
  - Large nu: more training points can be outside (looser boundary)

Let us visualize the effect of each.

In [ ]:
# Effect of gamma: side-by-side decision boundaries
gamma_values = [0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for ax, gamma in zip(axes, gamma_values):
    # Train on normal data only
    model_g = OneClassSVM(kernel='rbf', nu=0.1, gamma=gamma)
    model_g.fit(X_normal_2d)
    
    # Create mesh
    margin = 1.0
    xx, yy = np.meshgrid(
        np.linspace(X_2d[:, 0].min() - margin, X_2d[:, 0].max() + margin, 200),
        np.linspace(X_2d[:, 1].min() - margin, X_2d[:, 1].max() + margin, 200)
    )
    Z = model_g.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # Plot
    ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
    ax.contourf(xx, yy, Z, levels=[0, Z.max()], colors='palegreen', alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkgreen')
    
    ax.scatter(X_normal_2d[:, 0], X_normal_2d[:, 1],
              c='steelblue', s=10, alpha=0.4)
    ax.scatter(X_anomaly_2d[:, 0], X_anomaly_2d[:, 1],
              c='crimson', s=40, marker='x', linewidths=1.5)
    ax.set_title(f'gamma = {gamma}')

plt.suptitle('Effect of gamma on Decision Boundary (nu=0.1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Exercise 6

**Task:** Create a similar side-by-side plot for different values of `nu`: `[0.01, 0.05, 0.1, 0.3]`, keeping `gamma = 2.0` fixed. Adapt the code from the gamma exploration above.

**Questions to answer (as comments or markdown):**
1. What happens when `gamma` is too large? Too small?
2. What happens when `nu` is too large? Too small?
3. How would you choose these hyperparameters in practice?

In [ ]:
# ============================================================
# EXERCISE 6: Visualize the effect of nu on the decision boundary
# TODO: Create a 1x4 subplot grid
# TODO: For each nu in [0.01, 0.05, 0.1, 0.3]:
#       - Train a OneClassSVM with kernel='rbf', gamma=2.0, and the given nu
#       - Plot the decision boundary using contourf and contour
#       - Overlay normal and anomaly points
#       - Set the subplot title to show the nu value
# ============================================================

nu_values_viz = [0.01, 0.05, 0.1, 0.3]

# TODO: Create figure and axes
# fig, axes = plt.subplots(____)

# TODO: Loop over nu values and plot each decision boundary
# for ax, nu in zip(axes, nu_values_viz):
#     ...

# plt.suptitle('Effect of nu on Decision Boundary (gamma=2.0)', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.show()

# ANSWERS:
# 1. gamma too large: ____
# 2. gamma too small: ____
# 3. nu too large: ____
# 4. nu too small: ____
# 5. In practice: ____

---

## 9. Reflection & Open Questions

### When to use One-Class SVM?

One-Class SVM is a strong choice when:
- You have **abundant normal data** but **few or no labelled anomalies**
- The anomalies are **diverse** and hard to model directly
- The data is **moderate-dimensional** (up to a few hundred features)

### Trade-offs in Practice

| Consideration | Details |
|---------------|----------|
| **False Positives** | Flagging normal transactions as fraud causes customer friction (blocked cards, manual review costs). |
| **False Negatives** | Missing actual fraud leads to financial losses. |
| **Threshold tuning** | The `decision_function` returns continuous scores; choosing where to draw the line is a **business decision**, not just a statistical one. |

### Limitations of One-Class SVM

- **Scalability:** Training complexity is roughly $O(n^2)$ to $O(n^3)$, which can be prohibitive for very large datasets (>100k samples).
- **Sensitivity to hyperparameters:** `nu` and `gamma` can dramatically change results. Cross-validation is tricky without labelled anomalies.
- **Feature engineering matters:** Like all kernel methods, garbage in = garbage out.

### Extensions & Alternatives

| Method | Key Idea | When to Prefer |
|--------|----------|----------------|
| **Isolation Forest** | Anomalies are easier to isolate (fewer random splits needed). | Large datasets, high dimensions |
| **Local Outlier Factor (LOF)** | Compares local density of a point to its neighbors. | When anomalies have different local densities |
| **Autoencoders** | Learn to reconstruct normal data; anomalies have high reconstruction error. | Very high-dimensional data (images, sequences) |
| **Gaussian Mixture Models** | Fit a probabilistic model; low-probability points are anomalies. | When normal data has multiple clear clusters |

---

## 10. References

1. **Scholkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J., & Williamson, R. C.** (2001). *Estimating the Support of a High-Dimensional Distribution.* Neural Computation, 13(7), 1443-1471.  
   The foundational paper introducing the One-Class SVM formulation.

2. **scikit-learn documentation** -- [OneClassSVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.OneClassSVM.html)  
   API reference and practical usage examples.

3. **scikit-learn User Guide** -- [Novelty and Outlier Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)  
   Comprehensive guide comparing One-Class SVM, Isolation Forest, LOF, and more.

4. **Course materials** -- Machine Learning and Analytics, SUTD.  
   Lecture slides and supplementary readings.

---

*End of Notebook. Good luck with the exercises!*